In [1]:
print("hi")

hi


In [1]:
import os
import re
import json
import pdfplumber

from PIL import Image
# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)

COLUMN_HEADERS = [
    "service_dates",
    "proc_code",
    "billed",
    "allowed",
    "provider_discount",
    "not_covered",    
    "deductible",
    "coinsurance",
    "copay",
    "payment"
    ]

SCHEMA = {
    "tables": [
        {
            "table_index": "",
            "rows": [
                {
                    "provider": "",
                    "service_dates": "",
                    "proc_code": "",
                    "billed": "",
                    "allowed": "",
                    "provider_discount": "",
                    "not_covered": "",
                    "deductible": "",
                    "coinsurance": "",
                    "copay": "",
                    "payment": ""
                }
            ],
            "column_totals": {
                "billed": "",
                "allowed": "",
                "provider_discount": "",
                "not_covered": "",
                "deductible": "",
                "coinsurance": "",
                "copay": "",
                "payment": ""
            },
            "other_insurance_credits_or_adjustments": "",
            "total_payment_amount": "",
            "member_responsibility": ""
        }
    ]
}

# =========================================================
# EXTRACTION PROMPT
# =========================================================
PROMPT = """
You are an expert healthcare EOB (Explanation of Benefits) data extraction system.

Your task is to extract ALL table data accurately and return STRICT JSON ONLY.

-----------------------------------
🚨 CRITICAL TABLE RULES
-----------------------------------

HEADER EXTRACTION

Extract provider from the HEADER ABOVE THE TABLE.

provider =
The value immediately following "Provider:"

1. Extract ALL visible data rows exactly as they appear.

2. A VALID DATA ROW must contain:
   - service_dates
   AND
   - proc_code

3. SKIP rows where:
   - service_dates is missing
   OR
   - proc_code is missing

4. NEVER extract:
   - Column Totals row
   - Summary rows
   - Footer rows
   - Header rows
   - Payment summary rows

5. A DATA ROW exists ONLY between:
   "Service Dates"
   and
   "Column Totals"

6. ALWAYS preserve row count exactly as shown in image.

7. DO NOT MERGE ROWS under ANY condition:
   - Even if procedure code is same
   - Even if service dates are same
   - Even if values are identical
   - Even if rows look duplicated
   - Each visual row = one JSON row

8. DUPLICATE ROWS MUST BE PRESERVED.

9. If same proc_code appears multiple times
   with different service dates,
   extract them as separate rows.

10. If same proc_code appears multiple times
    with SAME service dates,
    still extract them as separate rows.

-----------------------------------
🚨 COLUMN TOTALS RULE
-----------------------------------

11. "Column Totals" row is NOT a data row.

12. DO NOT include "Column Totals" inside rows.

13. Extract totals ONLY inside:
    "column_totals"

-----------------------------------
🚨 COLUMN FILTERING RULES
-----------------------------------

14. IGNORE these columns completely:
    - Tooth #
    - Reason Code

15. Only extract these columns:
    - provider
    - service_dates
    - proc_code
    - billed
    - allowed
    - provider_discount
    - not_covered
    - deductible
    - coinsurance
    - copay
    - payment

-----------------------------------
🚨 PATIENT INFORMATION RULE
-----------------------------------

16. Extract patient_name separately.

17. Preserve patient name exactly as shown.

18. If patient name missing:
    return ""

-----------------------------------
🚨 SUMMARY SECTION RULE
-----------------------------------

19. Extract these values:
    - other_insurance_credits_or_adjustments
    - total_payment_amount
    - member_responsibility

20. These values are NOT table rows.

-----------------------------------
🚨 DATA PRESERVATION RULES
-----------------------------------

21. Preserve values EXACTLY as shown.

22. Preserve date format exactly.

23. Preserve procedure codes exactly.

24. Preserve row order exactly.

25. If value is missing:
    return ""

26. Never normalize values.

27. Never calculate values.

28. Never infer values.

-----------------------------------
🚨 CONFIDENCE RULES
-----------------------------------

EVERY EXTRACTED FIELD MUST USE THIS FORMAT:

{
    "value": "extracted value",
    "confidence": 0.95
}

NEVER return an extracted field as a plain string.

WRONG:

"proc_code": "D4341"

CORRECT:

"proc_code": {
    "value": "D4341",
    "confidence": 0.99
}

This applies to EVERY field in:
- patient_name
- provider
- every field in every row
- every field in column_totals
- other_insurance_credits_or_adjustments
- total_payment_amount
- member_responsibility

Confidence rules:

- confidence MUST be a number between 0.0 and 1.0.
- 1.0 = completely certain that the value was correctly read.
- 0.0 = value is missing, unreadable, or cannot be reliably extracted.
- Do not guess values.
- If a value cannot be reliably extracted:

{
    "value": "",
    "confidence": 0.0
}

- Confidence represents ONLY confidence in reading the value from the image.
- Do NOT calculate confidence based on the financial amount.
- Do NOT use validation results to determine field confidence.

-----------------------------------
🚨 OUTPUT FORMAT
-----------------------------------

Return ONLY valid JSON.

No explanation.
No markdown.
No extra text.

{
    "tables": [
        {
            "table_index": {
                "value": "",
                "confidence": 0.0
            },

            "patient_name": {
                "value": "",
                "confidence": 0.0
            },

            "provider": {
                "value": "",
                "confidence": 0.0
            },

            "rows": [
                {
                    "service_dates": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "proc_code": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "billed": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "allowed": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "provider_discount": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "not_covered": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "deductible": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "coinsurance": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "copay": {
                        "value": "",
                        "confidence": 0.0
                    },

                    "payment": {
                        "value": "",
                        "confidence": 0.0
                    }
                }
            ],

            "column_totals": {
                "billed": {
                    "value": "",
                    "confidence": 0.0
                },

                "allowed": {
                    "value": "",
                    "confidence": 0.0
                },

                "provider_discount": {
                    "value": "",
                    "confidence": 0.0
                },

                "not_covered": {
                    "value": "",
                    "confidence": 0.0
                },

                "deductible": {
                    "value": "",
                    "confidence": 0.0
                },

                "coinsurance": {
                    "value": "",
                    "confidence": 0.0
                },

                "copay": {
                    "value": "",
                    "confidence": 0.0
                },

                "payment": {
                    "value": "",
                    "confidence": 0.0
                }
            },

            "other_insurance_credits_or_adjustments": {
                "value": "",
                "confidence": 0.0
            },

            "total_payment_amount": {
                "value": "",
                "confidence": 0.0
            },

            "member_responsibility": {
                "value": "",
                "confidence": 0.0
            }
        }
    ]
}

-----------------------------------
🚨 FINAL CHECK
-----------------------------------

Before responding, verify:

1. Every extracted field contains:
   - "value"
   - "confidence"

2. No extracted field is a plain string.

3. Every row contains all required columns.

4. Column Totals are NOT included as normal rows.

5. Duplicate rows are preserved.

6. Rows missing service_dates OR proc_code are skipped.

7. Return ONLY valid JSON.

"""

# =========================================================
# IMAGE -> JSON EXTRACTION
# =========================================================
def extract_table_from_image(image_path, expected_rows=None):

    image = Image.open(image_path).convert("RGB")

    row_instruction = ""

    if expected_rows is not None:
        row_instruction = f"""

=========================================================
IMPORTANT PHYSICAL ROW COUNT
=========================================================

An independent PDF analysis detected EXACTLY {expected_rows}
physical service row(s) in this image.

You MUST return exactly {expected_rows} objects inside "rows".

IMPORTANT:
- Count PHYSICAL service rows.
- Each visible service row = exactly one JSON row.
- Do NOT count the Column Totals row.
- Do NOT count the header row.
- Do NOT create extra rows.
- Do NOT duplicate a physical row.
- Do NOT merge physical rows.
- Two physically separate rows MAY contain identical values.
- If two physical rows have identical values, KEEP BOTH.
- Never remove a row just because its values are identical to another row.
- Preserve the original visual row order.

The number of objects inside "rows" MUST be exactly {expected_rows}.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": PROMPT + row_instruction
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=2048
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    # REMOVE MARKDOWN IF MODEL RETURNS IT
    generated_text = generated_text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    return generated_text


def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS  # ✅ ordered list

    for table in parsed_output.get("tables", []):

        # =========================
        # 🔹 CLEAN ROWS (STRICT + ORDERED)
        # =========================
        cleaned_rows = []

        for row in table.get("rows", []):

            cleaned_row = {}

            # enforce order + remove unwanted keys
            for col in allowed_columns:
                cleaned_row[col] = row.get(col, "")

            service_dates = cleaned_row.get("service_dates", "")
            proc_code = cleaned_row.get("proc_code", "")

            if service_dates not in ["", None] and proc_code not in ["", None]:
                cleaned_rows.append(cleaned_row)

        table["rows"] = cleaned_rows

        # =========================
        # 🔹 CLEAN COLUMN TOTALS (STRICT + ORDERED)
        # =========================
        totals = table.get("column_totals", {})
        ordered_totals = {}

        for col in allowed_columns:
            if col not in ["service_dates", "proc_code"]:
                ordered_totals[col] = totals.get(col, "")

        table["column_totals"] = ordered_totals

        # =========================
        # 🔹 ENSURE REQUIRED FIELDS EXIST
        # =========================
        table["other_insurance_credits_or_adjustments"] = table.get(
            "other_insurance_credits_or_adjustments", ""
        )
        table["total_payment_amount"] = table.get(
            "total_payment_amount", ""
        )
        table["member_responsibility"] = table.get(
            "member_responsibility", ""
        )

    return parsed_output

# =========================================================
# MAIN PIPELINE
# =========================================================
def check_claim_denied(page):

    """
    Logic:
    Search ONLY before:
    'Attention Non-contracted Medicare Providers'

    If denied / denial keywords exist before that section,
    return True else False
    """

    full_text = page.extract_text()

    if not full_text:
        return False

    # -----------------------------------------------------
    # LIMIT SEARCH AREA
    # -----------------------------------------------------

    stop_keyword = "For Claim Submissions and ReSubmissions:"

    if stop_keyword in full_text:

        full_text = full_text.split(stop_keyword)[0]

    searchable_text = full_text.lower()

    # -----------------------------------------------------
    # DENIAL KEYWORDS
    # -----------------------------------------------------

    denial_keywords = [

        "denied",
        "denial",
    ]

    # -----------------------------------------------------
    # SEARCH
    # -----------------------------------------------------

    for keyword in denial_keywords:

        if keyword in searchable_text:

            print(f"❌ Claim Denied Keyword Found: {keyword}")

            return "denied"

    return "not denied"

#date_formate_change
def normalize_date_of_service(obj):
    """
    Converts:
        10/21-10/21/2022  -> 10/21/2022
        08/04-08/04/2023  -> 08/04/2023

    Works recursively for dicts/lists.
    """

    pattern = re.compile(r"^(\d{2}/\d{2})-\d{2}/\d{2}/(\d{4})$")

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "date_of_service" and isinstance(value, str):
                m = pattern.match(value.strip())
                if m:
                    obj[key] = f"{m.group(1)}/{m.group(2)}"
            else:
                normalize_date_of_service(value)

    elif isinstance(obj, list):
        for item in obj:
            normalize_date_of_service(item)

    return obj

def extract_provider(page):

    text = page.extract_text() or ""
    text_lower = text.lower()

    match = re.search(r"provider:\s*(.*?)\s*subscriber", text_lower)

    if match:
        start, end = match.span(1)
        return text[start:end].strip()

    return ""

def normalize_currency_values(obj):
    """
    Converts:
        "$1,234.50" -> "1234.50"
        "$50.00"     -> "50.00"
        ""           -> ""

    Works recursively for dict/list.
    """

    if isinstance(obj, dict):
        for key, value in obj.items():
            obj[key] = normalize_currency_values(value)

    elif isinstance(obj, list):
        return [
            normalize_currency_values(item)
            for item in obj
        ]

    elif isinstance(obj, str):

        value = obj.strip()

        if value == "":
            return ""

        # currency detection
        if "$" in value:
            value = (
                value
                .replace("$", "")
                .replace(",", "")
                .strip()
            )

            try:
                return f"{float(value):.2f}"
            except ValueError:
                return obj

    return obj

def crop_all_eob_tables(
    pdf_path,
    output_dir="EOB_OUTPUT/EMI_health", company_name = "EMI Health"
):
    

    pdf_full_name = os.path.basename(pdf_path)

    # pdf_name = pdf_file.split("_")[-1].split(".")[0]
    
    pdf_dir = os.path.basename(
        pdf_path
    ).split(".")[0].split("_")[-1]

    output_dir = os.path.join(
        output_dir,
        pdf_dir
    )

    os.makedirs(output_dir, exist_ok=True)

    all_patients = []
    confidence_results = []

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(
            pdf.pages,
            start=1
        ):
            is_denied = check_claim_denied(page)

            print(f"Denied Status : {is_denied}")
            words = page.extract_words()
            patient_name = extract_patient_name(page)
            provider = extract_provider(page)

            start_positions = []
            end_positions = []

            # =================================================
            # FIND ALL TABLES
            # =================================================

            for i, w in enumerate(words):

                text = w["text"].strip()

                # START MARKER
                if text == "Patient":

                    combined = " ".join(
                        word["text"]
                        for word in words[i:i+4]
                    )

                    if "Patient Account" in combined:

                        start_positions.append(
                            w["top"] - 10
                        )

                # END MARKER
                if text == "Member":

                    combined = " ".join(
                        word["text"]
                        for word in words[i:i+3]
                    )

                    if "Member Responsibility" in combined:

                        end_positions.append(
                            w["bottom"] + 10
                        )

            # =================================================
            # VALIDATION
            # =================================================

            table_count = min(
                len(start_positions),
                len(end_positions)
            )

            if table_count == 0:

                # results.append({
                #     "page": page_num,
                #     "status": "failed",
                #     "reason": "no tables found"
                # })

                continue

            # =================================================
            # PROCESS EACH TABLE
            # =================================================

            for idx in range(table_count):

                start_y = start_positions[idx]
                end_y = end_positions[idx]

                bbox = (
                    0,
                    start_y,
                    page.width,
                    end_y
                )

                cropped_page = page.crop(bbox)

                image_path = os.path.join(
                    output_dir,
                    f"page_{page_num}_table_{idx+1}.png"
                )

                cropped_page.to_image(
                    resolution=300
                ).save(image_path)

                # =============================================
                # LLM EXTRACTION
                # =============================================

                try:

                    # =============================================
                    # COUNT PHYSICAL SERVICE ROWS FIRST
                    # =============================================
                    expected_rows = count_service_rows(
                        page,
                        start_y,
                        end_y
                    )

                    print(
                        f"📌 Expected physical service rows: {expected_rows}"
                    )


                    # =============================================
                    # VLM EXTRACTION
                    # =============================================
                    llm_output = extract_table_from_image(
                        image_path,
                        expected_rows=expected_rows
                    )

                    parsed_output = json.loads(
                        llm_output
                    )

                    # =========================================================
                    # CONFIDENCE CALCULATION + UNWRAP
                    # =========================================================

                    for i, table in enumerate(
                        parsed_output.get("tables", [])
                    ):

                        # Calculate confidence BEFORE removing
                        # the value + confidence wrappers
                        table_model_confidence = (
                            calculate_model_confidence(table)
                        )

                        # Remove:
                        # {"value": "...", "confidence": 0.99}
                        #
                        # and keep:
                        # "..."
                        table = _unwrap_vlm_output(table)

                        # Keep model confidence internally
                        table["_model_confidence"] = (
                            table_model_confidence
                        )

                        parsed_output["tables"][i] = table

                    # =========================================================
                    # NORMAL EXISTING PROCESSING
                    # =========================================================

                    parsed_output = normalize_currency_values(
                        parsed_output
                    )

                    parsed_output = enforce_schema(
                        parsed_output
                    )

                    # =============================================
                    # ENFORCE EXPECTED ROW COUNT
                    # =============================================
                    for table in parsed_output.get("tables", []):

                        rows = table.get("rows", [])

                        if expected_rows is not None and len(rows) > expected_rows:

                            print(
                                f"⚠️ VLM returned {len(rows)} rows, "
                                f"but PDF detected {expected_rows} rows."
                            )

                            # IMPORTANT:
                            # Do NOT deduplicate.
                            # Identical physical rows are valid.
                            rows = rows[:expected_rows]

                            table["rows"] = rows



                    for table in parsed_output.get("tables", []):
                        table["patient_name"] = patient_name
                        table["provider"] = provider

                    # expected_rows = count_service_rows(page, start_y, end_y)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        row_count_ok = validate_service_row_count(
                            page,
                            start_y,
                            end_y,      
                            table,
                            t_idx
                        )
                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        is_valid, log, errors, total_fields = validate_eob_table(table, t_idx)
                        if not row_count_ok:
                            is_valid = False
                            errors = errors + [{"type": "row_count_mismatch"}]                      


                    for table in parsed_output.get("tables", []):

                        structured_table = {
                            "EOB_ID": pdf_dir,
                            "claim_status": is_denied,
                            "patient_name": table.get("patient_name", ""),
                            "provider": table.get("provider", ""),
                            "rows": table.get("rows", []),
                            "totals": table.get("column_totals", {}),
                            "validation": {"status": is_valid, "errors": errors},
                            "_expected_rows": expected_rows,
                            "_total_fields": total_fields,
                            "_model_confidence": table.get("_model_confidence", 0.0),
                            # "other_insurance_credits_or_adjustments": table.get("other_insurance_credits_or_adjustments", ""),
                            # "total_payment_amount": table.get("total_payment_amount", ""),
                            # "member_responsibility": table.get("member_responsibility", "")
                        }

                        print(f"✅ Completed Table {idx + 1} on Page {page_num}")

                    date_of_service = ""

                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("service_dates", "")

                    provider = structured_table.get("provider", "")

                    for row in structured_table.get("rows", []):

                        for col in ["service_dates", "provider"]:
                            row.pop(col, None)


                    patient_data = {
                        "patient_name": structured_table.get("patient_name", ""),
                        "provider": provider,
                        "date_of_service": date_of_service,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("totals", {}),
                        "validation": structured_table.get("validation"),
                        "_expected_rows": structured_table.get("_expected_rows", 0),
                        "_total_fields": structured_table.get("_total_fields", 0),
                        "_model_confidence": structured_table.get("_model_confidence", 0.0),
                        # "other_insurance_credits_or_adjustments": structured_table.get("other_insurance_credits_or_adjustments", ""),
                        # "total_payment_amount": structured_table.get("total_payment_amount", ""),
                        # "member_responsibility": structured_table.get("member_responsibility", "")
                    }                    
                    normalize_date_of_service(patient_data)
                    all_patients.append(patient_data)
                    confidence_results.append(patient_data) 

                    

                    

                except Exception as e:
                    print(f"error:str{e}")

    confidence_score = calculate_eob_confidence(confidence_results)



    final_output = [
        {
            "eob_id": pdf_dir,
            "file_name":pdf_full_name,
            "payor": "EMI Health",
            "claim_status": is_denied,
            "confidence_score": confidence_score,
            "patients": all_patients
        }
    ]           

    success_path, failed_path = save_split_output(
                    final_output,
                    company_name=company_name,
                    pdf_name=pdf_dir,
                    pdf_path=pdf_path,
                    cropped_dir=output_dir,
                )
            
    print(f"\n📁 Cropped images : {output_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output

#------------------- PARSE AMOUNT --------------------#

def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())

#------------------- VALIDATE TOTAL AMOUNT --------------------#




import re

def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()
    row_positions = []

    for w in words:
        text = w["text"].strip()

        match = re.search(r"\bD\d{4}\b", text)

        if match:
            y = float(w["top"])

            if region_top <= y <= region_bottom:
                row_positions.append(y)

    row_positions.sort()

    grouped_rows = []
    threshold = 3

    for y in row_positions:
        if not grouped_rows:
            grouped_rows.append(y)
        else:
            if abs(y - grouped_rows[-1]) > threshold:
                grouped_rows.append(y)

    return len(grouped_rows)

def extract_patient_name(page):

    words = page.extract_words()
    #patient_name = extract_patient_name(page)

    for i, w in enumerate(words):
        text = w["text"].strip()

        if text == "Patient:":
            # Next word(s) are the name
            name_parts = []

            # Collect next 2–4 words (depends on name length)
            for j in range(i + 1, i + 5):
                if j < len(words):
                    next_word = words[j]["text"].strip()

                    # stop if we hit another label
                    if ":" in next_word:
                        break

                    name_parts.append(next_word)

            return " ".join(name_parts)

    return ""

def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    totals = table.get("column_totals", {})

    if not rows:
        return False, "", []

    computed_totals = {
        "billed": round(sum(parse_amount(r.get("billed", "")) for r in rows), 2),
        "allowed": round(sum(parse_amount(r.get("allowed", "")) for r in rows), 2),
        "provider_discount": round(sum(parse_amount(r.get("provider_discount", "")) for r in rows), 2),
        "not_covered": round(sum(parse_amount(r.get("not_covered", "")) for r in rows), 2),   
        "deductible": round(sum(parse_amount(r.get("deductible", "")) for r in rows), 2),
        "coinsurance": round(sum(parse_amount(r.get("coinsurance", "")) for r in rows), 2),
        "copay": round(sum(parse_amount(r.get("copay", "")) for r in rows), 2),
        "payment": round(sum(parse_amount(r.get("payment", "")) for r in rows), 2)
    }
    total_fields = len(computed_totals)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    computed_member_resp = round(
    sum(parse_amount(r.get("not_covered", "")) for r in rows) +
    sum(parse_amount(r.get("deductible", "")) for r in rows) +
    sum(parse_amount(r.get("coinsurance", "")) for r in rows) +
    sum(parse_amount(r.get("copay", "")) for r in rows),
    2
    )

    # extracted_member_resp = round(
    #     parse_amount(table.get("member_responsibility", "")),
    #     2
    # )

    # if computed_member_resp == extracted_member_resp:
    #     icon = "✅"
    #     status = "match"
    # else:
    #     icon = "❌"
    #     status = "MISMATCH"
    #     has_error = True

    #     errors.append({
    #         "field": "member_responsibility",
    #         "computed": computed_member_resp,
    #         "extracted": extracted_member_resp
    #     })

    # line = f"{icon} {'member_responsibility':25s} computed={computed_member_resp:<10} | extracted={extracted_member_resp:<10} {status}"
    # print(line)
    # result_validation += "\n" + line


    # computed_total_payment = round(
    #     sum(parse_amount(r.get("billed", "")) for r in rows) -
    #     parse_amount(table.get("member_responsibility", "")),
    #     2
    # )

    # extracted_total_payment = round(
    #     parse_amount(table.get("total_payment_amount", "")),
    #     2
    # )

    # if computed_total_payment == extracted_total_payment:
    #     icon = "✅"
    #     status = "match"
    # else:
    #     icon = "❌"
    #     status = "MISMATCH"
    #     has_error = True

    #     # errors.append({
    #     #     "field": "total_payment_amount",
    #     #     "computed": computed_total_payment,
    #     #     "extracted": extracted_total_payment
    #     # })

    # line = f"{icon} {'total_payment_amount':25s} computed={computed_total_payment:<10} | extracted={extracted_total_payment:<10} {status}"
    # print(line)
    # result_validation += "\n" + line    
    # print("-" * 75)

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields
    

def validate_service_row_count(page, start_y, end_y, table, table_index):

    detected_count = count_service_rows(page, start_y, end_y)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("proc_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    if detected_count == extracted_count:
        icon = "✅"
        status = "match"
    else:
        icon = "❌"
        status = "MISMATCH"

    print(f"{icon} row_count        detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)


    return detected_count == extracted_count


def save_json(results, output_path="emi_results.json"):
    
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Saved results to {output_path}")

W0901 18:59:36.236000 3488500 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 18:59:36.250000 3488500 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Emi_health/pdf_files/Pmt_EOP_838537978.pdf")

Denied Status : not denied
📌 Expected physical service rows: 4


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count        detected=4     | extracted=4     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed                    computed=328.0      | extracted=328.0      match
✅ allowed                   computed=177.03     | extracted=177.03     match
✅ provider_discount         computed=0.0        | extracted=0.0        match
✅ not_covered               computed=150.97     | extracted=150.97     match
✅ deductible                computed=50.0       | extracted=50.0       match
✅ coinsurance               computed=0.0        | extracted=0.0        match
✅ copay                     computed=0.0        | extracted=0.0        match
✅ payment                   computed=127.03     | extracted=127.03     match
✅ [Table 1] Validation PASSED

✅ Completed Tab

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count        detected=1     | extracted=1     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed                    computed=445.0      | extracted=445.0      match
✅ allowed                   computed=230.11     | extracted=230.11     match
✅ provider_discount         computed=0.0        | extracted=0.0        match
✅ not_covered               computed=214.89     | extracted=214.89     match
✅ deductible                computed=0.0        | extracted=0.0        match
✅ coinsurance               computed=46.02      | extracted=46.02      match
✅ copay                     computed=0.0        | extracted=0.0        match
✅ payment                   computed=184.09     | extracted=184.09     match
✅ [Table 1] Validation PASSED

✅ Completed Tab

[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count        detected=2     | extracted=2     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed                    computed=734.0      | extracted=734.0      match
✅ allowed                   computed=0.0        | extracted=0.0        match
✅ provider_discount         computed=0.0        | extracted=0.0        match
✅ not_covered               computed=734.0      | extracted=734.0      match
✅ deductible                computed=0.0        | extracted=0.0        match
✅ coinsurance               computed=0.0        | extracted=0.0        match
✅ copay                     computed=0.0        | extracted=0.0        match
✅ payment                   computed=0.0        | extracted=0.0        match
✅ [Table 1] Validation PASSED

✅ Completed Tab

[{'eob_id': '838537978',
  'file_name': 'Pmt_EOP_838537978.pdf',
  'payor': 'EMI Health',
  'claim_status': 'not denied',
  'confidence_score': 97.0,
  'patients': [{'patient_name': 'Joel Hoffman',
    'provider': 'Alina Halusic',
    'date_of_service': '02/18/2026',
    'services': [{'proc_code': 'D0150',
      'billed': '139.00',
      'allowed': '74.38',
      'provider_discount': '0.00',
      'not_covered': '64.62',
      'deductible': '50.00',
      'coinsurance': '0.00',
      'copay': '0.00',
      'payment': '24.38'},
     {'proc_code': 'D0274',
      'billed': '99.00',
      'allowed': '53.09',
      'provider_discount': '0.00',
      'not_covered': '45.91',
      'deductible': '0.00',
      'coinsurance': '0.00',
      'copay': '0.00',
      'payment': '53.09'},
     {'proc_code': 'D0220',
      'billed': '45.00',
      'allowed': '24.78',
      'provider_discount': '0.00',
      'not_covered': '20.22',
      'deductible': '0.00',
      'coinsurance': '0.00',
      'copay': 

## Test 1

In [ ]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/EMI/Pmt_EOP_848325397.pdf")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Denied Status : not denied


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📊 Row Count Validation [Table 1]
----------------------------------------------------------------------
✅ row_count        detected=5     | extracted=5     match
----------------------------------------------------------------------

🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ billed                    computed=500.0      | extracted=500.0      match
✅ allowed                   computed=192.0      | extracted=192.0      match
✅ provider_discount         computed=0.0        | extracted=0.0        match
✅ not_covered               computed=308.0      | extracted=308.0      match
✅ deductible                computed=100.0      | extracted=100.0      match
✅ coinsurance               computed=0.0        | extracted=0.0        match
✅ copay                     computed=0.0        | extracted=0.0        match
✅ payment                   computed=92.0       | extracted=92.0       match
✅ [Table 1] Validation PASSED

✅ Completed Tab

[{'eob_id': '848325397',
  'payor': 'EMI Health',
  'claim_status': 'not denied',
  'confidence_score': 99.0,
  'patients': [{'patient_name': 'Kara West',
    'provider': 'Duc Tang',
    'date_of_service': '03/05/2026',
    'services': [{'proc_code': 'D0150',
      'billed': '148.00',
      'allowed': '53.00',
      'provider_discount': '0.00',
      'not_covered': '95.00',
      'deductible': '53.00',
      'coinsurance': '0.00',
      'copay': '0.00',
      'payment': '0.00'},
     {'proc_code': 'D0274',
      'billed': '104.00',
      'allowed': '42.00',
      'provider_discount': '0.00',
      'not_covered': '62.00',
      'deductible': '42.00',
      'coinsurance': '0.00',
      'copay': '0.00',
      'payment': '0.00'},
     {'proc_code': 'D0220',
      'billed': '49.00',
      'allowed': '19.00',
      'provider_discount': '0.00',
      'not_covered': '30.00',
      'deductible': '5.00',
      'coinsurance': '0.00',
      'copay': '0.00',
      'payment': '14.00'},
     {'proc_c

: 